# Entrenamiento del detector

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ninguna")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import subprocess

from google.colab import userdata

REPO = "github.com/Riv2oo4/Trabajo-De-Graduacion.git"
RAMA = "oe4-oe5-y-reproducibilidad"

# el token sale de los secretos de Colab y se pasa por subprocess, que no
# imprime el comando: así no queda escrito en el notebook ni en la salida
token = userdata.get("GITHUB_TOKEN")

# si una corrida anterior dejó el proceso parado dentro de /content/repo,
# borrarlo deja getcwd sin directorio y ningún subproceso vuelve a arrancar
os.chdir("/content")

!rm -rf /content/repo
subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA,
                f"https://x-access-token:{token}@{REPO}", "/content/repo"], check=True)

# sin estos dos módulos la corrida sale sin semilla fija, sin procedencia y sin
# intervalos, que es lo que la tabla vieja no tenía
for modulo in ("reproducibilidad.py", "intervalos.py"):
    assert os.path.exists(f"/content/repo/analisis/{modulo}"), \
        f"el clon de {RAMA} no trae {modulo}: revisar que el clon no haya fallado"
%cd /content/repo/analisis

## Datos

In [ ]:
import os

DRIVE = "/content/drive/MyDrive/Corpus-tesis"
ZIP_REALES = f"{DRIVE}/corpus_real.zip"
MANIPULADAS = f"{DRIVE}/manipuladas"
RESULTADOS = f"{DRIVE}/resultados"

assert os.path.exists(ZIP_REALES), ZIP_REALES
assert os.path.isdir(MANIPULADAS), MANIPULADAS
os.makedirs(RESULTADOS, exist_ok=True)

In [ ]:
# en disco local: leer 500 imágenes por época desde Drive es lento
!mkdir -p /content/reales && unzip -q -o "{ZIP_REALES}" -d /content/reales
!ls /content/reales

In [ ]:
import csv

REALES = "/content/reales/real"
CATALOGO = "/content/repo/corpus/catalogo.csv"

filas = list(csv.DictReader(open(CATALOGO)))
faltan = [f["archivo"] for f in filas
          if not os.path.exists(f"{REALES}/{f['archivo']}")
          or not os.path.exists(f"{MANIPULADAS}/{f['archivo']}")]
print(f"parejas: {len(filas)}  faltantes: {len(faltan)}")
assert not faltan, faltan[:5]

## Preentrenamiento sobre CIFAKE

In [ ]:
import kagglehub

CIFAKE = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")
!ls "{CIFAKE}"

In [ ]:
# la raíz debe contener train/ y test/
RAIZ_CIFAKE = CIFAKE
PESOS_CIFAKE = f"{RESULTADOS}/cifake.pt"

!python preentrenamiento_cifake.py --datos "{RAIZ_CIFAKE}" --salida "{PESOS_CIFAKE}" --epocas 5

## Comparación de inicializaciones

In [ ]:
CMD = (
    "python experimento_oe1.py"
    f" --catalogo {CATALOGO}"
    f" --reales {REALES}"
    f" --manipuladas '{MANIPULADAS}'"
    f" --pesos-cifake '{PESOS_CIFAKE}'"
    f" --salida '{RESULTADOS}/detector_oe1.csv'"
    f" --puntos-de-control '{RESULTADOS}/puntos_de_control'"
    " --epocas 10 --lote 32"
)
!{CMD}

## Resultados

In [ ]:
import pandas as pd

tabla = pd.read_csv(f"{RESULTADOS}/detector_oe1.csv")
NIVELES = ["todos", "obvio", "normal", "sutil"]

# global, con su intervalo
tabla[tabla.conjunto.eq("prueba") & tabla.nivel.eq("todos")][
    ["rama", "agregacion", "exactitud_balanceada", "auc", "auc_ic_bajo",
     "auc_ic_alto", "f1", "localizacion", "localizacion_ic_bajo",
     "localizacion_ic_alto"]
].sort_values(["agregacion", "auc"], ascending=[True, False])

In [ ]:
# por nivel de manipulación
tabla[tabla.conjunto.eq("prueba") & tabla.nivel.isin(NIVELES[1:])].pivot(
    index=["rama", "agregacion"], columns="nivel", values="exactitud_balanceada")

In [ ]:
# subconjunto guatemalteco por separado, y referente de la edición
tabla[tabla.conjunto.eq("prueba") & tabla.nivel.str.contains(":", na=False)][
    ["rama", "agregacion", "nivel", "n_imagenes", "exactitud_balanceada", "auc",
     "localizacion"]].sort_values(["nivel", "agregacion"])

In [ ]:
tabla[tabla.conjunto.eq("estres") & tabla.nivel.eq("todos")][
    ["rama", "agregacion", "exactitud_balanceada", "auc", "localizacion"]
].sort_values(["agregacion", "auc"], ascending=[True, False])

In [ ]:
from reproducibilidad import leer_procedencia

# confirma que la tabla quedó atribuible a esta corrida; es lo que el notebook
# del OE4 exige antes de empezar
d = leer_procedencia(f"{RESULTADOS}/detector_oe1.csv")
print(d["fecha"], "| commit", d["git"]["commit"], "| semilla", d["semilla"],
      "| GPU", d["entorno"]["gpu"])
print("parejas por partición:", d["parametros"]["parejas"])